In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("graficos", exist_ok=True)

In [ ]:
np.random.seed(42)
n = 600

regiao_loja = {
    "Loja Centro": "Sudeste",
    "Loja Norte": "Norte",
    "Loja Sul": "Sul",
    "Loja Leste": "Nordeste",
    "Loja Oeste": "Centro-Oeste",
}

produtos = {
    "Notebook": ("Eletrônicos", 3500),
    "Monitor": ("Eletrônicos", 900),
    "Webcam": ("Eletrônicos", 260),
    "Mouse": ("Acessórios", 80),
    "Teclado": ("Acessórios", 150),
    "Headset": ("Acessórios", 220),
    "Cadeira": ("Móveis", 700),
    "Mesa": ("Móveis", 600),
}

vendas_base = pd.DataFrame({
    "data": np.random.choice(pd.date_range("2025-01-01", "2025-12-31"), n),
    "loja": np.random.choice(list(regiao_loja), n),
    "produto": np.random.choice(list(produtos), n),
    "quantidade": np.random.randint(1, 10, n).astype(float),
})
vendas_base["categoria"] = vendas_base["produto"].map(lambda p: produtos[p][0])
vendas_base["preco_unitario"] = vendas_base["produto"].map(lambda p: produtos[p][1]) * np.random.uniform(0.9, 1.1, n)
vendas_base["regiao"] = vendas_base["loja"].map(regiao_loja)

vendas_base.loc[vendas_base.sample(15, random_state=1).index, "quantidade"] = np.nan
vendas_base.loc[vendas_base.sample(15, random_state=2).index, "preco_unitario"] = np.nan

vendas_base.to_csv("vendas.csv", index=False)

In [ ]:
vendas = pd.read_csv("vendas.csv")
vendas["data"] = pd.to_datetime(vendas["data"])
vendas

In [ ]:
vendas["faturamento"] = vendas["quantidade"] * vendas["preco_unitario"]

In [ ]:
vendas.info()

In [ ]:
vendas.isnull().sum()

In [ ]:
vendas.describe()

In [ ]:
vendas["quantidade"] = vendas["quantidade"].fillna(vendas["quantidade"].mean())
vendas["preco_unitario"] = vendas["preco_unitario"].fillna(vendas["preco_unitario"].mean())
vendas["faturamento"] = vendas["quantidade"] * vendas["preco_unitario"]
vendas.isnull().sum()

In [ ]:
faturamento_loja = (
    vendas.groupby("loja")["faturamento"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
faturamento_loja

In [ ]:
ticket_medio = (
    vendas.groupby("loja")["faturamento"].sum()
    / vendas.groupby("loja")["faturamento"].count()
).reset_index(name="ticket_medio")
ticket_medio

In [ ]:
top_produtos = (
    vendas.groupby("produto")["quantidade"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
top_produtos

In [ ]:
faturamento_mensal = vendas.groupby(vendas["data"].dt.to_period("M"))["faturamento"].sum()

mes_maior = faturamento_mensal.idxmax().strftime("%m/%Y")
mes_menor = faturamento_mensal.idxmin().strftime("%m/%Y")

print(f"Maior faturamento: {mes_maior} (R$ {faturamento_mensal.max():,.2f})")
print(f"Menor faturamento: {mes_menor} (R$ {faturamento_mensal.min():,.2f})")

In [ ]:
vendas["mes"] = vendas["data"].dt.month
vendas.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=faturamento_loja, x="loja", y="faturamento", hue="loja", palette="viridis", legend=False, ax=ax)
ax.set_title("Faturamento total por loja em 2025")
ax.set_xlabel("Loja")
ax.set_ylabel("Faturamento (R$)")
plt.tight_layout()
plt.savefig("graficos/faturamento_por_loja.png", dpi=150)
plt.show()

In [ ]:
resumo_mensal = vendas.groupby("mes")["faturamento"].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=resumo_mensal, x="mes", y="faturamento", marker="o", color="tab:blue", ax=ax)
for x, y in zip(resumo_mensal["mes"], resumo_mensal["faturamento"]):
    ax.annotate(f"{y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
ax.set_xticks(range(1, 13))
ax.set_title("Evolução mensal do faturamento em 2025")
ax.set_xlabel("Mês")
ax.set_ylabel("Faturamento (R$)")
plt.tight_layout()
plt.savefig("graficos/evolucao_mensal.png", dpi=150)
plt.show()

In [ ]:
faturamento_regiao = (
    vendas.groupby("regiao")["faturamento"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    faturamento_regiao["faturamento"],
    labels=faturamento_regiao["regiao"],
    autopct="%1.1f%%",
    colors=sns.color_palette("Set2"),
    startangle=90,
)
ax.set_title("Participação de cada região no faturamento total")
plt.tight_layout()
plt.savefig("graficos/faturamento_por_regiao.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=vendas, x="quantidade", y="faturamento", hue="categoria", palette="Set1", ax=ax)
ax.set_title("Relação entre quantidade vendida e faturamento por categoria")
ax.set_xlabel("Quantidade de itens")
ax.set_ylabel("Faturamento (R$)")
ax.legend(title="Categoria")
plt.tight_layout()
plt.savefig("graficos/quantidade_vs_faturamento.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=vendas, x="categoria", y="preco_unitario", hue="categoria", palette="pastel", legend="full", ax=ax)
ax.set_title("Distribuição do preço unitário por categoria de produto")
ax.set_xlabel("Categoria")
ax.set_ylabel("Preço unitário (R$)")
ax.legend(title="Categoria")
plt.tight_layout()
plt.savefig("graficos/preco_por_categoria.png", dpi=150)
plt.show()

In [ ]:
correlacao = vendas[["quantidade", "preco_unitario", "faturamento", "mes"]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlação entre as variáveis numéricas")
plt.tight_layout()
plt.savefig("graficos/correlacao.png", dpi=150)
plt.show()

In [ ]:
sns.set_style("whitegrid")
sns.set_palette(sns.color_palette("deep"))

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(20, 11))

sns.barplot(data=faturamento_loja, x="loja", y="faturamento", hue="loja", legend=False, ax=axs[0, 0])
axs[0, 0].set_title("Faturamento total por loja")
axs[0, 0].set_xlabel("Loja")
axs[0, 0].set_ylabel("Faturamento (R$)")
axs[0, 0].tick_params(axis="x", rotation=20)

sns.lineplot(data=resumo_mensal, x="mes", y="faturamento", marker="o", ax=axs[0, 1])
axs[0, 1].set_xticks(range(1, 13))
axs[0, 1].set_title("Evolução mensal do faturamento")
axs[0, 1].set_xlabel("Mês")
axs[0, 1].set_ylabel("Faturamento (R$)")

axs[0, 2].pie(
    faturamento_regiao["faturamento"],
    labels=faturamento_regiao["regiao"],
    autopct="%1.1f%%",
    startangle=90,
)
axs[0, 2].set_title("Participação das regiões no faturamento")

sns.scatterplot(data=vendas, x="quantidade", y="faturamento", hue="categoria", ax=axs[1, 0])
axs[1, 0].set_title("Quantidade vendida x faturamento")
axs[1, 0].set_xlabel("Quantidade de itens")
axs[1, 0].set_ylabel("Faturamento (R$)")
axs[1, 0].legend(title="Categoria")

sns.boxplot(data=vendas, x="categoria", y="preco_unitario", hue="categoria", legend="full", ax=axs[1, 1])
axs[1, 1].set_title("Preço unitário por categoria")
axs[1, 1].set_xlabel("Categoria")
axs[1, 1].set_ylabel("Preço unitário (R$)")

sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="coolwarm", ax=axs[1, 2])
axs[1, 2].set_title("Correlação entre variáveis numéricas")

plt.tight_layout()
plt.savefig("graficos/painel_completo.png", dpi=150)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=faturamento_loja, x="loja", y="faturamento", hue="loja", legend=False, ax=axs[0])
axs[0].set_title("Faturamento total por loja")
axs[0].set_xlabel("Loja")
axs[0].set_ylabel("Faturamento (R$)")
axs[0].tick_params(axis="x", rotation=20)

sns.barplot(data=faturamento_regiao, x="regiao", y="faturamento", hue="regiao", legend=False, ax=axs[1])
axs[1].set_title("Faturamento total por região")
axs[1].set_xlabel("Região")
axs[1].set_ylabel("Faturamento (R$)")
axs[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("graficos/comparativo_loja_regiao.png", dpi=150)
plt.show()

In [ ]:
resumo = faturamento_loja.merge(ticket_medio, on="loja")
resumo["regiao"] = resumo["loja"].map(regiao_loja)
resumo["participacao_pct"] = resumo["faturamento"] / resumo["faturamento"].sum() * 100
resumo["observacao"] = resumo.apply(
    lambda l: f"{l['loja']} ({l['regiao']}) responde por {l['participacao_pct']:.1f}% do faturamento, com ticket médio de R$ {l['ticket_medio']:,.2f}",
    axis=1,
)

geral = pd.DataFrame([{
    "loja": "Geral",
    "faturamento": vendas["faturamento"].sum(),
    "observacao": (
        f"Maior faturamento em {mes_maior} e menor em {mes_menor}; "
        f"produto mais vendido: {top_produtos.loc[0, 'produto']}; "
        f"correlação quantidade x faturamento: {correlacao.loc['quantidade', 'faturamento']:.2f}"
    ),
}])

resumo_analitico = pd.concat([resumo, geral], ignore_index=True)
resumo_analitico.to_csv("resumo_analitico.csv", index=False, sep=";")
resumo_analitico

In [ ]:
os.listdir("graficos")